In [1]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import heapq
import itertools
import time
from abc import ABC, abstractmethod
from collections import deque

# ==========================================
# 1. CORE ENGINE (AdvancedOrderBook)
# ==========================================

class AdvancedOrderBook:
    def __init__(self):
        self.bids = []  
        self.asks = []  
        self.trades = []
        self.order_id_counter = itertools.count() 

    def submit_order(self, side, qty, price=None, order_type='limit'):
        if order_type == 'market':
            limit_price = float('inf') if side == 'buy' else 0
        else:
            limit_price = price

        remaining_qty = self.match(side, qty, limit_price)

        if remaining_qty > 0 and order_type == 'limit':
            entry_id = next(self.order_id_counter)
            if side == 'buy':
                heapq.heappush(self.bids, [-limit_price, entry_id, remaining_qty])
            else:
                heapq.heappush(self.asks, [limit_price, entry_id, remaining_qty])
            
        return remaining_qty

    def match(self, side, qty, limit_price):
        remaining_qty = qty
        while remaining_qty > 0:
            if side == 'buy':
                if not self.asks: break
                best_price = self.asks[0][0]
                if limit_price < best_price: break
                best_order = self.asks[0]
            else:
                if not self.bids: break
                best_price = -self.bids[0][0] 
                if limit_price > best_price: break
                best_order = self.bids[0]

            trade_qty = min(remaining_qty, best_order[2])
            exec_price = best_order[0] if side == 'buy' else -best_order[0]

            self.trades.append({
                'price': exec_price,
                'qty': trade_qty,
                'timestamp': time.time(),
                'side': side,
                'aggressor': 'market' if limit_price == float('inf') else 'limit'
            })

            remaining_qty -= trade_qty
            best_order[2] -= trade_qty

            if best_order[2] == 0:
                if side == 'buy': heapq.heappop(self.asks)
                else: heapq.heappop(self.bids)
                    
        return remaining_qty

# ==========================================
# 2. ANALYTICS ENGINE
# ==========================================

class AnalyticsEngine:
    def __init__(self):
        self.tape = []      

    def log_trade(self, timestamp, price, qty, buyer_id, seller_id):
        self.tape.append({
            'timestamp': timestamp,
            'price': price,
            'qty': qty,
            'buyer_id': buyer_id,
            'seller_id': seller_id
        })

    def get_tape_dataframe(self):
        df = pd.DataFrame(self.tape)
        if not df.empty:
            df['timestamp'] = pd.to_datetime(df['timestamp'])
            df.set_index('timestamp', inplace=True)
        return df

# ==========================================
# 3. AGENTS (Noise + Momentum)
# ==========================================

class Agent(ABC):
    def __init__(self, agent_id, cash, inventory):
        self.agent_id = agent_id
        self.cash = cash
        self.inventory = inventory
        
    @abstractmethod
    def get_action(self, market_snapshot, fair_value=None):
        pass

class NoiseTrader(Agent):
    """
    Day 7 Agent: Follows hidden 'fair_value' with noise.
    Provides background liquidity.
    """
    def __init__(self, agent_id, cash, inventory, arrival_rate=0.5, volatility=0.02):
        super().__init__(agent_id, cash, inventory)
        self.arrival_rate = arrival_rate
        self.volatility = volatility 

    def get_action(self, market_snapshot, fair_value):
        if np.random.random() > self.arrival_rate: return None
        side = 'buy' if np.random.random() > 0.5 else 'sell'
        noise = np.random.normal(0, self.volatility * fair_value)
        order_price = round(fair_value + noise, 2)
        qty = np.random.randint(1, 5) # Small lots
        
        return {'agent_id': self.agent_id, 'side': side, 'price': order_price, 'qty': qty, 'type': 'limit'}

class MomentumTrader(Agent):
    """
    Day 8 Agent: Trend Follower.
    Strategy: 
      - Compute Simple Moving Average (SMA) of last N prices.
      - If Price > SMA + threshold: BUY (Betting on uptrend).
      - If Price < SMA - threshold: SELL (Betting on downtrend).
    """
    def __init__(self, agent_id, cash, inventory, lookback=20, threshold=0.5):
        super().__init__(agent_id, cash, inventory)
        self.lookback = lookback
        self.threshold = threshold
        # Internal memory of prices to calculate SMA without peeking at future
        self.price_history = deque(maxlen=lookback)

    def get_action(self, market_snapshot, fair_value=None):
        """
        Ignores 'fair_value'. Only looks at 'mid_price' history.
        """
        mid_price = market_snapshot.get('mid_price')
        
        # If no price exists yet (market start), do nothing
        if mid_price is None: return None
        
        # Update internal history
        self.price_history.append(mid_price)
        
        # Need full history to compute valid SMA
        if len(self.price_history) < self.lookback:
            return None

        # 1. Compute Technical Indicator (SMA)
        sma = sum(self.price_history) / len(self.price_history)
        
        # 2. Decision Logic (Trend Following)
        # We place Aggressive Limit orders (Market-like) to ensure execution
        qty = 10 # Momentum traders bet bigger than noise traders
        
        if mid_price > sma + self.threshold:
            # UPTREND DETECTED -> BUY
            return {
                'agent_id': self.agent_id,
                'side': 'buy',
                'price': round(mid_price + 1.0, 2), # Aggressive buy
                'qty': qty,
                'type': 'limit'
            }
        elif mid_price < sma - self.threshold:
            # DOWNTREND DETECTED -> SELL
            return {
                'agent_id': self.agent_id,
                'side': 'sell',
                'price': round(mid_price - 1.0, 2), # Aggressive sell
                'qty': qty,
                'type': 'limit'
            }
            
        return None # No clear trend

# ==========================================
# 4. SIMULATION LOOP (Scenario: Pump & Dump)
# ==========================================

# Configuration
SEED = 50 # Changed seed to ensure trend formation
NUM_STEPS = 3000
OUTPUT_FILENAME = "day8_momentum_report.pdf"

def generate_fair_value_series(steps, start_price=100.0, drift=0.0, vol=0.0005):
    prices = [start_price]
    for _ in range(steps):
        change = np.random.normal(drift, vol)
        new_price = prices[-1] * (1 + change)
        prices.append(new_price)
    return prices

def process_data(df_tape):
    if df_tape.empty: return pd.DataFrame()
    ohlc = df_tape['price'].resample('1min').ohlc()
    ohlc['volume'] = df_tape['qty'].resample('1min').sum()
    ohlc.dropna(inplace=True)
    return ohlc

def generate_report(ohlc_data, fair_values):
    print(f"Generating report: {OUTPUT_FILENAME}...")
    with PdfPages(OUTPUT_FILENAME) as pdf:
        # Page 1: Price Action
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Plot Hidden Fair Value
        times = pd.date_range(start="2025-01-01 09:30", periods=len(fair_values), freq="S")
        ax.plot(times, fair_values, color='orange', alpha=0.3, label='Fair Value (Hidden)')

        # Plot Candles
        up = ohlc_data[ohlc_data.close >= ohlc_data.open]
        down = ohlc_data[ohlc_data.close < ohlc_data.open]
        ax.bar(up.index, up.close - up.open, bottom=up.open, width=0.0005, color='green')
        ax.vlines(up.index, up.low, up.high, color='green')
        ax.bar(down.index, down.close - down.open, bottom=down.open, width=0.0005, color='red')
        ax.vlines(down.index, down.low, down.high, color='red')
        
        ax.set_title("Day 8: Momentum Agents (Trend Following Instability)")
        ax.legend()
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
        pdf.savefig(fig)
        plt.close()
        
        # Page 2: Volume
        fig2, ax2 = plt.subplots(figsize=(10, 6))
        ax2.bar(ohlc_data.index, ohlc_data['volume'], width=0.0005, color='blue', alpha=0.6)
        ax2.set_title("Volume Profile (Check for spikes)")
        pdf.savefig(fig2)
        plt.close()

def run_simulation():
    print(f"--- Starting Day 8 Simulation (Momentum) ---")
    random.seed(SEED)
    np.random.seed(SEED)
    
    engine = AdvancedOrderBook()
    analytics = AnalyticsEngine()
    
    # 1. Fair Value (Brownian Motion)
    fair_values = generate_fair_value_series(NUM_STEPS, start_price=100.0)
    
    # 2. Agents: Mix of Noise (Liquidity) and Momentum (Speculators)
    # 20 Noise Traders to provide the "ocean"
    noise_agents = [NoiseTrader(i, 100000, 1000) for i in range(20)]
    
    # 5 Momentum Traders (The "Whales" that chase trends)
    momentum_agents = [MomentumTrader(100+i, 100000, 0, lookback=30) for i in range(5)]
    
    all_agents = noise_agents + momentum_agents
    
    current_time = datetime(2025, 1, 1, 9, 30)
    
    print(f"Simulating {len(all_agents)} agents over {NUM_STEPS} steps...")
    
    for i in range(NUM_STEPS):
        current_time += timedelta(seconds=1)
        current_fair_value = fair_values[i]
        
        # Snapshot
        best_bid = -engine.bids[0][0] if engine.bids else None
        best_ask = engine.asks[0][0] if engine.asks else None
        
        # Fallback for mid_price if book is empty
        mid_price = (best_bid + best_ask) / 2 if (best_bid and best_ask) else current_fair_value
        
        snapshot = {'best_bid': best_bid, 'best_ask': best_ask, 'mid_price': mid_price}
        
        random.shuffle(all_agents)
        
        for agent in all_agents:
            # Noise traders use fair_value, Momentum traders use snapshot history
            order = agent.get_action(snapshot, current_fair_value)
            
            if order:
                engine.submit_order(order['side'], order['qty'], order['price'], order['type'])

        while engine.trades:
            trade = engine.trades.pop(0)
            analytics.log_trade(current_time, trade['price'], trade['qty'], "Ag", "Pas")

    print("Simulation complete.")
    return analytics.get_tape_dataframe(), fair_values

if __name__ == "__main__":
    df_tape, fv_data = run_simulation()
    if not df_tape.empty:
        df_ohlc = process_data(df_tape)
        if not df_ohlc.empty:
            generate_report(df_ohlc, fv_data)
            print(f"Done. Check {OUTPUT_FILENAME}")
    else:
        print("No trades.")

--- Starting Day 8 Simulation (Momentum) ---
Simulating 25 agents over 3000 steps...
Simulation complete.
Generating report: day8_momentum_report.pdf...
Done. Check day8_momentum_report.pdf


C:\Users\Asus User\AppData\Local\Temp\ipykernel_22708\1247177822.py:223: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  times = pd.date_range(start="2025-01-01 09:30", periods=len(fair_values), freq="S")
